# 2 : Analyse des éléments de symétrie

Dans cette section, nous analysons l'effet de trois opérations de symétrie distinctes (en excluant l'identité) du groupe d'espace $P\bar{4}m2$. Conformément aux consignes, chaque opération sera appliquée à un atome différent de la maille de notre matériau : l'Aluminium (Al), le Gallium (Ga) et l'Azote (N).

In [10]:
import numpy as np
from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.cif import CifWriter
from jupyter_jsmol import JsmolView
from IPython.display import display
from ipywidgets import Layout

# 1. Récupération de la structure
API_KEY = "vKJsFu0jdhLy7CJj5Mwar6S68kxgMc3n"
MP_ID = "mp-1008556"

with MPRester(API_KEY) as mpr:
    structure = mpr.get_structure_by_material_id(MP_ID)

# 2. Passage en maille conventionnelle pour l'analyse des symétries
analyzer = SpacegroupAnalyzer(structure)
conv_struc = analyzer.get_conventional_standard_structure()
analyze_struc = SpacegroupAnalyzer(conv_struc)

# Liste complète des 8 opérations de symétrie du groupe ponctuel
sym = analyze_struc.get_symmetry_operations()

# 3. Création d'un fichier CIF local pour JSmol
cif_filename = "AlGaN2_conv.cif"
CifWriter(conv_struc).write_file(cif_filename)

# 4. Identification robuste des coordonnées d'un atome de chaque espèce
sites = conv_struc.sites
Al_coord = [site.frac_coords for site in sites if site.specie.symbol == 'Al'][0]
Ga_coord = [site.frac_coords for site in sites if site.specie.symbol == 'Ga'][0]
N_coord =  [site.frac_coords for site in sites if site.specie.symbol == 'N'][0]

# 5. Sélection automatique et incassable des 3 opérations
S1 = None; S2 = None; S3 = None
for op in sym:
    # On supprime les espaces pour éviter les bugs de version pymatgen
    xyz = op.as_xyz_str().replace(" ", "")
    
    if xyz == "-x,-y,z":
        S1 = op  # Rotation d'ordre 2
    elif xyz == "x,-y,z" or xyz == "-x,y,z":
        S2 = op  # Plan Miroir
    elif xyz == "-y,x,-z" or xyz == "y,-x,-z":
        S3 = op  # Roto-inversion

# 6. Calcul des nouvelles coordonnées
Al_sym = S1.operate(Al_coord)
Ga_sym = S2.operate(Ga_coord)
N_sym = S3.operate(N_coord)

print("Les 3 opérations ont été correctement identifiées et appliquées !")

Les 3 opérations ont été correctement identifiées et appliquées !


### 2.1. Rotation d'ordre 2 sur l'Aluminium (Al)

La première opération étudiée est une rotation d'ordre 2 ($C_2$) autour de l'axe $z$, appliquée à l'atome d'Aluminium. Mathématiquement, cette opération inverse les coordonnées $x$ et $y$ tout en conservant $z$. Elle est définie par l'équation matricielle suivante : 

$$\begin{pmatrix} x' \\ y' \\ z' \end{pmatrix} = \begin{pmatrix} -1 & 0 & 0 \\ 0 & -1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \begin{pmatrix} x \\ y \\ z \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \\ 0 \end{pmatrix}$$

Le vecteur de translation $\tau$ est nul. Le déterminant de la matrice de rotation vaut $1$ : il s'agit donc d'une opération de première espèce (rotation propre) qui conserve la chiralité de l'environnement atomique.


In [11]:
print("--- OPÉRATION DE SYMÉTRIE 1 (Rotation) --- \n")
print(f"L'atome de départ est l'Aluminium (Al) aux coordonnées : {Al_coord}\n")
print(f"Application de l'opération {S1.as_xyz_str()} :\n")
print(S1.rotation_matrix, " * ", Al_coord, " + ", S1.translation_vector)
print(f"\nL'atome d'arrivée a pour coordonnées : {Al_sym}\n")

# Visualisation JSmol
jsmol_1 = JsmolView.from_file(cif_filename)
display(jsmol_1)
jsmol_1.script('select aluminum; color lightgray; set spacefill 0.5')
jsmol_1.script('select gallium; color lightgreen')
jsmol_1.script('select nitrogen; color lightblue')
# L'index JSmol est souvent l'index Python + 1 (ici 4+1 = 5)
jsmol_1.script('draw symop 5 {atomno=1}')

--- OPÉRATION DE SYMÉTRIE 1 (Rotation) --- 

L'atome de départ est l'Aluminium (Al) aux coordonnées : [0.5 0.5 0.5]

Application de l'opération -x, -y, z :

[[-1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]  *  [0.5 0.5 0.5]  +  [0. 0. 0.]

L'atome d'arrivée a pour coordonnées : [-0.5 -0.5  0.5]



JsmolView(layout=Layout(align_self='stretch', height='400px'))

### 2.2. Réflexion (Plan Miroir) sur le Gallium (Ga)

La deuxième opération est une réflexion (plan miroir $m$) par rapport au plan $xz$, appliquée à l'atome de Gallium. Cette opération préserve les coordonnées $x$ et $z$ mais inverse le signe de $y$. Son équation associée est la suivante : 

$$\begin{pmatrix} x' \\ y' \\ z' \end{pmatrix} = \begin{pmatrix} 1 & 0 & 0 \\ 0 & -1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \begin{pmatrix} x \\ y \\ z \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \\ 0 \end{pmatrix}$$

Le déterminant de cette matrice vaut $-1$. Il s'agit d'une opération de deuxième espèce, qui inverse la chiralité.

In [12]:
print("--- OPÉRATION DE SYMÉTRIE 2 (Miroir) --- \n")
print(f"L'atome de départ est le Gallium (Ga) aux coordonnées : {Ga_coord}\n")
print(f"Application de l'opération {S2.as_xyz_str()} :\n")
print(S2.rotation_matrix, " * ", Ga_coord, " + ", S2.translation_vector)
print(f"\nL'atome d'arrivée a pour coordonnées : {Ga_sym}\n")

# Visualisation JSmol
jsmol_2 = JsmolView.from_file(cif_filename)
display(jsmol_2)
jsmol_2.script('select aluminum; color lightgray')
jsmol_2.script('select gallium; color lightgreen; set spacefill 0.5')
jsmol_2.script('select nitrogen; color lightblue')
# L'index JSmol est souvent l'index Python + 1 (ici 8+1 = 9)
jsmol_2.script('draw symop 9 {atomno=2}')

--- OPÉRATION DE SYMÉTRIE 2 (Miroir) --- 

L'atome de départ est le Gallium (Ga) aux coordonnées : [0. 0. 0.]

Application de l'opération x, -y, z :

[[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]  *  [0. 0. 0.]  +  [0. 0. 0.]

L'atome d'arrivée a pour coordonnées : [0. 0. 0.]



JsmolView(layout=Layout(align_self='stretch', height='400px'))

### 2.3. Roto-inversion sur l'Azote (N)

La troisième et dernière opération étudiée est une roto-inversion d'ordre 4 ($\bar{4}$) appliquée à l'atome d'Azote. C'est l'opération caractéristique du groupe ponctuel de ce matériau ($-42m$). Elle combine une rotation de 90° autour de l'axe $z$ suivie d'une inversion. Elle est définie par :

$$\begin{pmatrix} x' \\ y' \\ z' \end{pmatrix} = \begin{pmatrix} 0 & -1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & -1 \end{pmatrix} \begin{pmatrix} x \\ y \\ z \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \\ 0 \end{pmatrix}$$

Le déterminant de cette matrice vaut également $-1$. Il s'agit donc d'une opération de deuxième espèce modifiant la chiralité du site.

In [13]:
print("--- OPÉRATION DE SYMÉTRIE 3 (Roto-inversion) --- \n")
print(f"L'atome de départ est l'Azote (N) aux coordonnées : {N_coord}\n")
print(f"Application de l'opération {S3.as_xyz_str()} :\n")
print(S3.rotation_matrix, " * ", N_coord, " + ", S3.translation_vector)
print(f"\nL'atome d'arrivée a pour coordonnées : {N_sym}\n")

# Visualisation JSmol
jsmol_3 = JsmolView.from_file(cif_filename)
display(jsmol_3)
jsmol_3.script('select aluminum; color lightgray')
jsmol_3.script('select gallium; color lightgreen')
jsmol_3.script('select nitrogen; color lightblue; set spacefill 0.5')
# L'index JSmol est souvent l'index Python + 1 (ici 2+1 = 3)
jsmol_3.script('draw symop 3 {atomno=3}')

--- OPÉRATION DE SYMÉTRIE 3 (Roto-inversion) --- 

L'atome de départ est l'Azote (N) aux coordonnées : [0.5        0.         0.73826481]

Application de l'opération -y, x, -z :

[[ 0. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0. -1.]]  *  [0.5        0.         0.73826481]  +  [0. 0. 0.]

L'atome d'arrivée a pour coordonnées : [ 0.          0.5        -0.73826481]



JsmolView(layout=Layout(align_self='stretch', height='400px'))